In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

batch_size = 128
learning_rate = 1e-3
epochs = 5

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


using device: cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.72MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.6MB/s]


In [5]:
#slightly tweaked CNN, not really the LeNet
class LeNet(nn.Module):
    def __init__(self, num_classes = 10):
        super().__init__()

        #input : (B,1,28,28)
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1), # (B,6,24,24)
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2), # (B,6,12,12)

            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1), # (B,16,8,8)
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2) # (B,16,4,4)
        )

        self.classifier = nn.Sequential( #std MLP
            nn.Flatten(), # (B,16*4*4 = 256)
            nn.Linear(in_features=16*4*4, out_features=120),
            nn.ReLU(),
            nn.Linear(120,84),
            nn.ReLU(),
            nn.Linear(84, num_classes)
        )
    def forward (self, x):
      x = self.features(x)  #CNN - feature extractor
      x = self.classifier(x)
      return x

model = LeNet(num_classes = 10).to(device)
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

LeNet(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=120, bias=True)
    (2): ReLU()
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): ReLU()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)


In [9]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    return epoch_loss, epoch_acc

In [7]:
#Eval function
@torch.no_grad()
def evaluate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    return epoch_loss, epoch_acc

In [10]:
#training loop
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Epoch 1/5, Train Loss: 0.5563, Train Acc: 0.8264, Test Loss: 0.1913, Test Acc: 0.9395
Epoch 2/5, Train Loss: 0.1528, Train Acc: 0.9528, Test Loss: 0.0986, Test Acc: 0.9700
Epoch 3/5, Train Loss: 0.0962, Train Acc: 0.9707, Test Loss: 0.0823, Test Acc: 0.9738
Epoch 4/5, Train Loss: 0.0769, Train Acc: 0.9762, Test Loss: 0.0564, Test Acc: 0.9821
Epoch 5/5, Train Loss: 0.0629, Train Acc: 0.9808, Test Loss: 0.0543, Test Acc: 0.9824
